# Módulo 6: Ordenamiento, Estabilidad y Búsqueda en Arreglos

## 🎯 Objetivos de Aprendizaje
Al finalizar este cuaderno, serás capaz de:
1. Diferenciar entre el ordenamiento funcional e **inmutable** (`np.sort`) y la mutación directa en memoria (**in-place**, `arr.sort()`).
2. Controlar la dirección de ordenamiento en matrices 2D mediante `axis=0` (columnas) y `axis=1` (filas), así como invertir el orden (descendente).
3. Seleccionar el algoritmo de ordenamiento adecuado mediante `kind` (`'quicksort'`, `'mergesort'`, `'stable'`) para preservar el orden relativo en empates (**estabilidad**).
4. Utilizar `np.argsort()` para ordenar filas completas de una matriz basándose en una sola columna sin romper la correspondencia de los registros, y extraer valores distintos con `np.unique()`.

---

## 🗺️ Mapa Mental del Módulo

```text
                [ Ordenamiento y Búsqueda en NumPy ]
                                  │
      ┌───────────────────────────┼───────────────────────────┐
      ▼                           ▼                           ▼
1. ORDENAMIENTO BÁSICO      2. EJES Y ESTABILIDAD       3. ÍNDICES Y BÚSQUEDA
├── np.sort(A) -> Copia     ├── axis=0 (cada columna)   ├── np.argsort() (índices)
├── A.sort() -> In-place    ├── axis=1 (cada fila)      ├── Ordenar tabla por columna
└── Descendente: [::-1]     └── kind='stable' (empates) └── np.unique(return_counts)

In [3]:
# Importar numpy
import numpy as np

## 6.1 Ordenamiento Inmutable (`np.sort`) vs. In-Place (`arr.sort`)

NumPy ofrece dos formas principales de ordenar elementos de menor a mayor (orden ascendente):

| Método / Función | Comportamiento en Memoria | Valor de Retorno | Uso Recomendado |
| :--- | :--- | :--- | :--- |
| **`np.sort(arr)`** | **Inmutable:** Crea y devuelve una **copia ordenada**. El arreglo original permanece intacto. | Devuelve el nuevo `ndarray` ordenado. | Cuando necesitas conservar el orden cronológico o original de tus datos. |
| **`arr.sort()`** | **In-place (Mutable):** Reordena los elementos directamente dentro del bloque de memoria original. | Devuelve `None`. | Cuando trabajas con matrices muy grandes y deseas ahorrar memoria RAM. |

### Ordenamiento Descendente (Mayor a Menor)
NumPy no tiene un parámetro `reverse=True` como las listas de Python. Para ordenar de mayor a menor, se combina `np.sort()` con un *slicing* de paso negativo (`[::-1]`).

### Sintaxis Mínima

In [4]:
# Vector 1D: Tiempos (segundos) registrados por 10 corredores en una prueba
tiempos_carrera = np.array([12.4, 11.8, 13.1, 11.5, 12.9, 11.2, 14.0, 12.1, 11.9, 12.6])

# 1. Ordenamiento inmutable (np.sort): devuelve copia ordenada, original intacto
tiempos_ascendente = np.sort(tiempos_carrera)
print("Original intacto       :", tiempos_carrera)
print("Copia ascendente       :", tiempos_ascendente)

# 2. Ordenamiento descendente (mayor a menor) usando slicing [::-1]
tiempos_descendente = np.sort(tiempos_carrera)[::-1]
print("Copia descendente      :", tiempos_descendente)

# 3. Ordenamiento in-place (arr.sort()): modifica el arreglo directamente y devuelve None
copia_trabajo = tiempos_carrera.copy()
retorno = copia_trabajo.sort()
print("\nValor devuelto por .sort():", retorno)
print("Arreglo mutado in-place   :", copia_trabajo)

Original intacto       : [12.4 11.8 13.1 11.5 12.9 11.2 14.  12.1 11.9 12.6]
Copia ascendente       : [11.2 11.5 11.8 11.9 12.1 12.4 12.6 12.9 13.1 14. ]
Copia descendente      : [14.  13.1 12.9 12.6 12.4 12.1 11.9 11.8 11.5 11.2]

Valor devuelto por .sort(): None
Arreglo mutado in-place   : [11.2 11.5 11.8 11.9 12.1 12.4 12.6 12.9 13.1 14. ]


:::{warning} Trampa Común: Asignar `x = arr.sort()`
Un error clásico proveniente de programar sin verificar retornos es escribir `ordenado = tiempos_carrera.sort()`. Como `.sort()` opera *in-place* y devuelve `None`, la variable `ordenado` quedará vacía (`None`) y habrás destruido el orden original de `tiempos_carrera`. Usa siempre `ordenado = np.sort(tiempos_carrera)`.
:::

### 🧩 Micro-reto 6.1
Dado el vector `puntajes` con 10 calificaciones en la celda inferior:
1. Obtén el **Top 3 de las mejores calificaciones** (de mayor a menor) sin modificar el vector `puntajes` original.
2. Verifica mediante `np.array_equal()` que `puntajes` conserva exactamente su orden inicial.

In [6]:
puntajes = np.array([78, 92, 65, 88, 95, 71, 84, 99, 60, 89])

# Escribe tu solución aquí:

## 6.2 Ordenamiento por Ejes (`axis=0`, `axis=1`) y Estabilidad (`kind='stable'`)

Cuando aplicas `np.sort(matriz, axis=...)` sobre una matriz 2D, NumPy ordena cada fila o cada columna **de manera totalmente independiente**:
* **`axis=0`:** Ordena los valores de arriba hacia abajo dentro de cada columna.
* **`axis=1` (por defecto en 2D):** Ordena los valores de izquierda a derecha dentro de cada fila.
* **`axis=None`:** Aplana toda la matriz a un vector 1D y luego lo ordena.

### Estabilidad Algorítmica (`kind='stable'`)
Un algoritmo de ordenamiento es **estable** cuando, ante dos elementos con exactamente el mismo valor (un empate), **garantiza que conservarán el mismo orden relativo** que tenían antes de ser ordenados.
* Por defecto, NumPy usa `'quicksort'` (muy rápido, pero **no es estable**).
* Si necesitas preservar el orden de llegada en caso de empate (fundamental al ordenar por múltiples criterios sucesivos), especifica `kind='stable'` (o `kind='mergesort'`).

### Sintaxis Mínima

In [7]:
# Matriz 4x4: Tiempos de respuesta (ms) de 4 servidores (filas) en 4 pruebas (columnas)
latencias = np.array([
    [145, 110, 180, 125],
    [95,  130, 105, 115],
    [210, 160, 190, 175],
    [120, 115, 140, 110]
])

print("Matriz original (4x4):\n", latencias)

# 1. Ordenar cada COLUMNA de menor a mayor (axis=0)
print("\nOrdenado por columnas (axis=0):\n", np.sort(latencias, axis=0))

# 2. Ordenar cada FILA de menor a mayor (axis=1)
print("\nOrdenado por filas (axis=1):\n", np.sort(latencias, axis=1))

# 3. Ordenar filas de MAYOR a MENOR (descendente por axis=1)
desc_por_fila = np.sort(latencias, axis=1)[:, ::-1]
print("\nOrdenado por filas de mayor a menor:\n", desc_por_fila)

# 4. Demostración de estabilidad con np.argsort(..., kind='stable')
# Supongamos calificaciones enteras empatadas de 8 participantes en orden de inscripción (índices 0 al 7)
notas_empatadas = np.array([8, 10, 8, 9, 10, 8, 9, 10])
indices_estables = np.argsort(notas_empatadas, kind='stable')
print("\nNotas con empates:", notas_empatadas)
print("Índices ordenados establemente (respeta orden 0, 2, 5 para el 8):", indices_estables)

Matriz original (4x4):
 [[145 110 180 125]
 [ 95 130 105 115]
 [210 160 190 175]
 [120 115 140 110]]

Ordenado por columnas (axis=0):
 [[ 95 110 105 110]
 [120 115 140 115]
 [145 130 180 125]
 [210 160 190 175]]

Ordenado por filas (axis=1):
 [[110 125 145 180]
 [ 95 105 115 130]
 [160 175 190 210]
 [110 115 120 140]]

Ordenado por filas de mayor a menor:
 [[180 145 125 110]
 [130 115 105  95]
 [210 190 175 160]
 [140 120 115 110]]

Notas con empates: [ 8 10  8  9 10  8  9 10]
Índices ordenados establemente (respeta orden 0, 2, 5 para el 8): [0 2 5 3 6 1 4 7]


:::{warning} Trampa Crítica: Romper la Integridad de los Registros con `np.sort()` en Tablas
Si tienes una matriz donde cada fila es un estudiante y las columnas son `[ID, Edad, Nota]`, ejecutar `np.sort(matriz, axis=0)` ordenará los IDs, las edades y las notas **por separado**, mezclando la edad de un estudiante con la nota de otro.
**Regla:** `np.sort(matriz, axis=0)` solo debe usarse cuando cada columna representa una serie independiente (por ejemplo, simulaciones de Monte Carlo). Para ordenar una tabla manteniendo intactas sus filas, debes usar **`np.argsort()`** (Subtema 6.3).
:::

### 🧩 Micro-reto 6.2
Dada una matriz `ventas_vendedores` de tamaño $4 \times 5$ (4 vendedores $\times$ 5 meses) definida en la celda inferior:
1. Obtén una matriz donde los meses de cada vendedor (filas) estén ordenados de **menor a mayor** (`axis=1`).
2. Extrae únicamente las **2 mejores ventas mensuales** de cada vendedor (las dos últimas columnas de la matriz ordenada por filas).

In [9]:
ventas_vendedores = np.array([
    [340, 290, 410, 380, 310],
    [500, 480, 520, 490, 510],
    [190, 250, 210, 280, 230],
    [420, 390, 450, 400, 440]
])

# Escribe tu solución aquí:

## 6.3 Ordenamiento por Índices (`np.argsort`) y Búsqueda de Únicos (`np.unique`)

### 1. `np.argsort(arr)`: La Llave para Ordenar Tablas Relacionales
En lugar de devolver los valores ordenados, `np.argsort()` devuelve **los índices posicionales** que ordenarían el arreglo. Al pasar esos índices como *Fancy Indexing* sobre las filas de una matriz (`matriz[indices]`), reordenamos todos los registros completos sin mezclar sus columnas.

### 2. `np.unique(arr, return_counts=False, return_index=False)`
Encuentra todos los valores distintos de un arreglo y los devuelve ordenados de menor a mayor. Con los parámetros opcionales:
* `return_counts=True`: Devuelve también la frecuencia absoluta de cada valor único (ideal para histogramas discretos y conteo de clases).
* `return_index=True`: Devuelve el índice de la primera aparición de cada valor.

### Sintaxis Mínima

In [13]:
# Matriz 5x3: Registro de productos -> Columnas: [Código_ID, Precio ($), Stock_Unidades]
inventario = np.array([
    [101, 45.0, 12],
    [102, 15.5, 40],
    [103, 89.0, 5],
    [104, 22.0, 30],
    [105, 15.5, 18]
])
print("Matriz iventario:\n", inventario)

# 1. Extraemos la columna de Precio (índice 1) y obtenemos los índices que la ordenan
indices_por_precio = np.argsort(inventario[:, 1], kind='stable')
print("\nÍndices ordenados por precio:", indices_por_precio)

# 2. Aplicamos esos índices a las FILAS de la matriz completa
inventario_ordenado_precio = inventario[indices_por_precio]
print("\nInventario ordenado por Precio ascendente (filas intactas):\n", inventario_ordenado_precio)

# 3. Ordenar por Stock (columna 2) de MAYOR a MENOR
indices_stock_desc = np.argsort(inventario[:, 2])[::-1]
print("\nInventario ordenado por Stock descendente:\n", inventario[indices_stock_desc])

# 4. Búsqueda y conteo de frecuencias con np.unique sobre un vector de categorías/respuestas
encuesta = np.array([3, 5, 4, 5, 2, 5, 4, 3, 5, 1, 4, 5, 3, 4, 5])
print("\nVector encuesta:", encuesta)

valores_unicos, conteos = np.unique(encuesta, return_counts=True)

print("\nCalificaciones únicas encontradas:", valores_unicos)
print("Frecuencia de cada calificación   :", conteos)

Matriz iventario:
 [[101.   45.   12. ]
 [102.   15.5  40. ]
 [103.   89.    5. ]
 [104.   22.   30. ]
 [105.   15.5  18. ]]

Índices ordenados por precio: [1 4 3 0 2]

Inventario ordenado por Precio ascendente (filas intactas):
 [[102.   15.5  40. ]
 [105.   15.5  18. ]
 [104.   22.   30. ]
 [101.   45.   12. ]
 [103.   89.    5. ]]

Inventario ordenado por Stock descendente:
 [[102.   15.5  40. ]
 [104.   22.   30. ]
 [105.   15.5  18. ]
 [101.   45.   12. ]
 [103.   89.    5. ]]

Vector encuesta: [3 5 4 5 2 5 4 3 5 1 4 5 3 4 5]

Calificaciones únicas encontradas: [1 2 3 4 5]
Frecuencia de cada calificación   : [1 1 3 4 6]


:::{note} Restricción de Dimensionalidad en `np.unique`
Si aplicas `np.unique(matriz)` sin especificar el parámetro `axis`, NumPy aplanará automáticamente la matriz 2D a un vector 1D antes de buscar los elementos únicos. Si deseas encontrar **filas únicas** completas dentro de una matriz, debes pasar explícitamente `np.unique(matriz, axis=0)`.
:::

### 🧩 Micro-reto 6.3
Dada la matriz `estudiantes` de tamaño $5 \times 3$ cuyas columnas representan `[ID_Estudiante, Edad, Promedio_Final]` en la celda inferior:
1. Ordena la matriz completa de **mayor a menor según el `Promedio_Final` (columna índice 2)** utilizando `np.argsort()`, asegurándote de que cada estudiante conserve su `ID` y su `Edad` correctos.
2. Utiliza `np.unique()` con `return_counts=True` sobre la columna de `Edad` (columna índice 1) para determinar qué edades distintas existen en el grupo y cuántos estudiantes tienen cada edad.

In [ ]:
estudiantes = np.array([
    [1001, 20, 8.4],
    [1002, 22, 9.5],
    [1003, 20, 7.8],
    [1004, 21, 9.1],
    [1005, 22, 8.9]
])

# Escribe tu solución aquí:

## ✅ Resumen del Módulo 6
* Utiliza **`np.sort(arr)`** para obtener una copia ordenada sin alterar los datos originales; reserva `arr.sort()` para mutaciones *in-place* donde la memoria sea crítica.
* Para ordenar de mayor a menor, combina el ordenamiento con el *slicing* inverso **`[::-1]`**.
* Especifica **`kind='stable'`** cuando necesites conservar el orden relativo original entre elementos empatados.
* Nunca uses `np.sort(tabla, axis=0)` para ordenar registros tabulares; emplea **`np.argsort()`** sobre la columna clave e indexa la matriz completa (`tabla[idx]`).